# Train chiến lược H1: touch XGBoost + Direction gộp XGBoost / LSTM (`touch_direction_h1_v2`)

Quy trình (xem `report.md`):
1. **Dựng dữ liệu:** nến 1h, funding, vị thế 5m; nhãn touch; 92 feature touch (lean) + 37 feature Direction.
2. **Dev 2022–2024:** walk-forward theo quý, 5 seed.
   - Chọn tham số giao dịch trên lưới bằng dự báo XGBoost (≥ `min_trades_per_year` lệnh/năm), rồi khoá config.
   - Direction production = 0.5·XGBoost + 0.5·LSTM (`cfg.direction_blend`, theo `exp_direction_blend.ipynb`), trộn theo từng seed. Tham số giao dịch giữ như bản khoá trên XGBoost, đúng như thử nghiệm đã xác nhận.
3. **Test 2025–2026 (ngoài mẫu):** walk-forward với config đã khoá, kèm bản frozen.
4. **Xuất bundle live** `artifacts/1h_strategy_v2/models_final.joblib`, gồm XGBoost touch + Direction và LSTM Direction, mỗi model 5 seed.
   - LSTM train trên GPU của máy này. Bundle lưu LSTM dạng ONNX, nên container live dự báo bằng onnxruntime, **không cần torch**.
   - Khi xuất, mỗi seed được kiểm tra: dự báo ONNX phải trùng dự báo torch.
   - Kèm theo `backtest_trades.csv` và `walkforward_signals.csv`.

Kết quả từng quý × seed được cache ở `artifacts/1h_strategy_v2/folds`, nên XGBoost và LSTM đã train ở `compare_models` sẽ được dùng lại. **Train lại và xuất bundle mỗi quý.**

In [1]:
import os, sys
from dataclasses import replace
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "model_training").exists() else Path.cwd().parent
os.chdir(ROOT)                                   # mọi đường dẫn trong config tính từ thư mục gốc repo
sys.path.insert(0, str(ROOT / "model_training"))
import numpy as np
import pandas as pd
import strategy_h1 as st

pd.set_option("display.width", 200, "display.max_columns", 40)
cfg = st.StrategyConfig()
print(cfg)
D = st.build(cfg)
st.data_report(D)

StrategyConfig(csv_5m='historical_data/BTCUSDT_5m_all.csv', positioning_csv='historical_data/BTCUSDT_positioning_5m.csv', funding_csv='historical_data/BTCUSDT_funding.csv', output_dir='artifacts/1h_strategy_v2', data_start='2020-01-01', dev_end='2025-01-01', first_dev_quarter='2022Q1', first_test_quarter='2025Q1', last_test_quarter=None, val_quarters=2, val_es_frac=0.5, seeds=(1000, 1001, 1002, 1003, 1004), horizon_hours=8, lookback_bars=168, vol_span=168, slow_vol_span=720, b_mult=1.0, min_barrier=0.01, max_barrier=0.05, touch_pivot_reversal=0.0075, time_decay_halflife_days=540.0, touch_xgb={}, pivot_reversal=0.01, direction_xgb={}, direction_label='3class', xgb_device='cpu', model_kind='xgb', nn_params={}, nn_device='auto', nn_threads=4, direction_blend={'xgb': 0.5, 'lstm': 0.5}, touch_gate_days=14, dir_window_days=90, thr_min_frac=0.5, dir_balance=True, gate_order='touch_first', touch_top=0.2, dir_top=0.1, tp_mult=1.0, sl_mult=1.0, max_hold_hours=9, max_positions=5, min_gap_bars=1, 

c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:305: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  "chạm +b trước / đã chạm": g.apply(lambda x: x.up.sum() / max(x.touch.sum() - x.tie.sum(), 1)),


,mẫu,b TB %,tỉ lệ chạm,chạm +b trước / đã chạm,chạm 2 mức cùng nến
năm,,,,,
2020,2967,1.6732,0.5130,0.5405,0.0010
2021,8760,2.4413,0.5357,0.4860,0.0016
2022,8760,1.8204,0.4709,0.4646,0.0033
2023,8760,1.2818,0.4264,0.5159,0.0026
2024,8784,1.5341,0.4855,0.5053,0.0016
2025,8760,1.3123,0.4505,0.4849,0.0010
2026,6731,1.2741,0.4723,0.4820,0.0013


## 1. Dev: walk-forward theo quý 2022Q1 → 2024Q4 (XGBoost touch + Direction, LSTM Direction)

In [2]:
P_dev, R_dev = st.walk_forward_blend(D, cfg, "dev")
P_dev_xgb = st.walk_forward(D, cfg, "dev")[0]          # từ cache: dùng để chọn tham số giao dịch
R_dev

dev 2022Q1: AUC touch 0.610 | AUC direction 0.616 | train 7,303 | 0s
dev 2022Q2: AUC touch 0.695 | AUC direction 0.536 | train 9,511 | 0s
dev 2022Q3: AUC touch 0.605 | AUC direction 0.520 | train 11,719 | 0s
dev 2022Q4: AUC touch 0.689 | AUC direction 0.601 | train 13,879 | 0s
dev 2023Q1: AUC touch 0.664 | AUC direction 0.675 | train 16,063 | 0s
dev 2023Q2: AUC touch 0.692 | AUC direction 0.588 | train 18,271 | 0s
dev 2023Q3: AUC touch 0.751 | AUC direction 0.565 | train 20,479 | 0s
dev 2023Q4: AUC touch 0.721 | AUC direction 0.566 | train 22,639 | 0s
dev 2024Q1: AUC touch 0.718 | AUC direction 0.571 | train 24,823 | 0s
dev 2024Q2: AUC touch 0.749 | AUC direction 0.587 | train 27,031 | 0s
dev 2024Q3: AUC touch 0.735 | AUC direction 0.585 | train 29,239 | 0s
dev 2024Q4: AUC touch 0.778 | AUC direction 0.626 | train 31,423 | 0s
dev 2022Q1: AUC touch 0.582 | AUC direction 0.621 | train 7,303 | 0s
dev 2022Q2: AUC touch 0.669 | AUC direction 0.545 | train 9,511 | 0s
dev 2022Q3: AUC touch 0.

,quý,mẫu test,tỉ lệ chạm,b TB %,AUC touch,AUC touch của Direction,AUC direction (đã chạm),n train,cây touch TB,cây direction TB,giây,model
0,2022Q1,2160,0.508796,1.977232,0.610240,0.577686,0.615681,7303,34.6,80.8,0,xgb
1,2022Q2,2184,0.508242,2.095003,0.695119,0.646283,0.536022,9511,132.4,90.8,0,xgb
2,2022Q3,2208,0.501359,1.841925,0.605306,0.537233,0.519545,11719,267.2,104.0,0,xgb
3,2022Q4,2208,0.366395,1.373910,0.688918,0.545331,0.600528,13879,244.0,163.8,0,xgb
4,2023Q1,2160,0.438426,1.561316,0.664231,0.548610,0.674600,16063,137.0,110.6,0,xgb
5,2023Q2,2184,0.469780,1.268490,0.691816,0.577534,0.588218,18271,298.8,477.4,0,xgb
6,2023Q3,2208,0.320199,1.085569,0.751090,0.625746,0.564907,20479,342.4,529.2,0,xgb
7,2023Q4,2208,0.477808,1.217631,0.720851,0.614273,0.565900,22639,390.2,245.6,0,xgb
8,2024Q1,2184,0.482601,1.625150,0.718122,0.589823,0.571460,24823,305.8,880.4,0,xgb
9,2024Q2,2184,0.478480,1.417689,0.748740,0.662858,0.586948,27031,457.6,509.6,0,xgb


## 2. Chọn tham số giao dịch trên dev (dự báo XGBoost) và khoá config

In [3]:
G = st.grid_search(D, P_dev_xgb, cfg)
G.to_csv(Path(cfg.output_dir) / "grid_dev.csv", index=False)
best = st.select_best(G)
display(G.head(10))
best

Lưới 96 ô, 76 ô hợp lệ (>= 150 lệnh/năm), 33s


,touch_top,dir_top,entry_mode,max_positions,min_gap_bars,lệnh,R/lệnh,tổng R,n_2022,R_2022,totR_2022,long_2022,n_2023,R_2023,totR_2023,long_2023,n_2024,R_2024,totR_2024,long_2024,TB năm,năm kém,điểm,hợp lệ,điểm vùng
0,0.2,0.10,limit_pullback,5,1,603,0.178414,107.583764,196,-0.013001,-2.548286,0.494898,211,0.355751,75.063420,0.507109,196,0.178922,35.068631,0.535714,0.173890,-0.013001,0.080444,True,0.073339
1,0.2,0.10,next_open,5,1,607,0.172693,104.824697,197,-0.002668,-0.525644,0.497462,214,0.327495,70.083919,0.504673,196,0.179931,35.266422,0.535714,0.168252,-0.002668,0.082792,True,0.072794
2,0.2,0.10,limit_pullback,3,1,513,0.176842,90.720174,169,0.022469,3.797340,0.491124,170,0.319187,54.261787,0.494118,174,0.187707,32.661047,0.522989,0.176455,0.022469,0.099462,True,0.071718
3,0.2,0.10,next_open,3,1,519,0.162529,84.352694,170,0.025870,4.397900,0.494118,172,0.283598,48.778821,0.494186,177,0.176135,31.175973,0.519774,0.161868,0.025870,0.093869,True,0.065726
4,0.5,0.05,next_open,5,2,522,0.123829,64.638850,162,0.029028,4.702479,0.530864,176,0.245735,43.249440,0.545455,184,0.090690,16.686931,0.527174,0.121818,0.029028,0.075423,True,0.064713
5,0.5,0.05,limit_pullback,5,2,520,0.107344,55.818902,162,0.024043,3.894973,0.530864,175,0.228059,39.910277,0.548571,183,0.065648,12.013651,0.524590,0.105917,0.024043,0.064980,True,0.059205
6,0.3,0.10,limit_pullback,3,1,665,0.126419,84.068926,207,0.020213,4.184004,0.444444,220,0.208080,45.777577,0.536364,238,0.143308,34.107345,0.512605,0.123867,0.020213,0.072040,True,0.058866
7,0.3,0.10,next_open,5,2,556,0.148236,82.419083,170,-0.000946,-0.160781,0.482353,196,0.269999,52.919782,0.561224,190,0.156106,29.660083,0.531579,0.141720,-0.000946,0.070387,True,0.056622
8,0.2,0.20,limit_pullback,5,2,703,0.140214,98.570551,227,0.015103,3.428418,0.484581,257,0.220102,56.566288,0.486381,219,0.176145,38.575845,0.529680,0.137117,0.015103,0.076110,True,0.055659
9,0.3,0.10,limit_pullback,5,2,554,0.143391,79.438550,169,-0.007904,-1.335851,0.479290,196,0.276748,54.242654,0.561224,189,0.140380,26.531747,0.529101,0.136408,-0.007904,0.064252,True,0.055532


touch_top                   0.2
dir_top                     0.1
entry_mode       limit_pullback
max_positions                 5
min_gap_bars                  1
lệnh                        603
R/lệnh                 0.178414
tổng R               107.583764
n_2022                      196
R_2022                -0.013001
totR_2022             -2.548286
long_2022              0.494898
n_2023                      211
R_2023                 0.355751
totR_2023              75.06342
long_2023              0.507109
n_2024                      196
R_2024                 0.178922
totR_2024             35.068631
long_2024              0.535714
TB năm                  0.17389
năm kém               -0.013001
điểm                   0.080444
hợp lệ                     True
điểm vùng              0.073339
Name: 0, dtype: object

In [4]:
# kiểm tra quy trình chọn: tham số cho năm Y chỉ chọn bằng các năm trước Y
st.nested_selection(G, cfg)

,năm test,chọn bằng,touch_top,dir_top,max_positions,min_gap_bars,entry_mode,R/lệnh năm test,lệnh năm test,TB toàn lưới năm test
0,2023,2022,0.5,0.05,5,2,next_open,0.245735,176,0.192142
1,2024,2022-2023,0.5,0.05,5,2,next_open,0.090690,184,0.118541


In [5]:
# tham khảo: nếu chọn trên dev bằng Direction gộp thì ô tốt nhất là gì (không dùng để khoá)
G_blend = st.grid_search(D, P_dev, cfg)
G_blend.head(5)

Lưới 96 ô, 70 ô hợp lệ (>= 150 lệnh/năm), 22s


,touch_top,dir_top,entry_mode,max_positions,min_gap_bars,lệnh,R/lệnh,tổng R,n_2022,R_2022,totR_2022,long_2022,n_2023,R_2023,totR_2023,long_2023,n_2024,R_2024,totR_2024,long_2024,TB năm,năm kém,điểm,hợp lệ,điểm vùng
0,0.5,0.05,limit_pullback,3,1,515,0.196897,101.401804,156,0.092329,14.403351,0.455128,199,0.241942,48.146462,0.532663,160,0.242825,38.851991,0.550000,0.192365,0.092329,0.142347,True,0.105660
1,0.5,0.05,limit_pullback,5,1,648,0.200160,129.703750,195,0.054479,10.623462,0.476923,254,0.293853,74.638563,0.539370,199,0.223325,44.441725,0.552764,0.190552,0.054479,0.122516,True,0.103733
2,0.2,0.10,limit_pullback,5,1,571,0.209618,119.691808,198,0.039240,7.769467,0.494949,209,0.352535,73.679766,0.483254,164,0.233186,38.242575,0.573171,0.208320,0.039240,0.123780,True,0.103310
3,0.5,0.05,next_open,5,1,656,0.199162,130.650540,197,0.067387,13.275227,0.477157,258,0.286945,74.031713,0.538760,201,0.215640,43.343600,0.562189,0.189990,0.067387,0.128689,True,0.099198
4,0.5,0.05,next_open,3,1,521,0.194051,101.100537,156,0.082571,12.881119,0.455128,201,0.251998,50.651694,0.537313,164,0.229071,37.567724,0.548780,0.187880,0.082571,0.135226,True,0.095760


In [6]:
cfg_l = st.lock_config(cfg, best, G)
print("direction_blend:", cfg_l.direction_blend)
dev = st.evaluate(D, P_dev, cfg_l, "dev")
st.save_result(dev, cfg_l, "dev")
display(pd.Series(dev["summary"]))
display(dev["by_year"])
st.seed_table(D, P_dev, cfg_l, "dev")

direction_blend: {'xgb': 0.5, 'lstm': 0.5}


trades                                                               571
win_rate                                                        0.602452
R_per_trade                                                     0.209618
total_R                                                       119.691808
max_dd_R                                                       23.003517
long_share                                                      0.513135
maker_share                                                     0.630473
timeout_share                                                   0.295972
b_mean_pct                                                      1.498141
pct_per_trade                                                   0.271338
total_pct                                                     154.934063
R_by_year              {2022: 7.769466564024299, 2023: 73.67976616176...
pct_by_year            {2022: 21.610892083392745, 2023: 87.9197642297...
trades_by_year                         {2022: 198, 

,lệnh,R/lệnh,tổng R,thắng,tỉ lệ LONG,TP,SL,hết giờ,khớp limit,b TB %,% giá/lệnh,tổng % giá,lãi % vốn
năm,,,,,,,,,,,,,
2022,198,0.039,7.769,0.500,0.495,0.394,0.268,0.338,0.657,1.872,0.109,21.611,6.194
2023,209,0.353,73.680,0.694,0.483,0.555,0.144,0.301,0.569,1.226,0.421,87.920,105.303
2024,164,0.233,38.243,0.610,0.573,0.537,0.226,0.238,0.677,1.393,0.277,45.403,44.436


,lệnh,R/lệnh,tổng R,% giá/lệnh,tổng % giá,lãi % vốn,R/lệnh 2022,R/lệnh 2023,R/lệnh 2024
seed,,,,,,,,,
1000,577.0000,0.2257,130.2053,0.2998,172.9763,250.1262,0.0527,0.3455,0.2633
1001,582.0000,0.1925,112.0110,0.2696,156.8998,191.2514,0.0212,0.3451,0.1711
1002,579.0000,0.2093,121.1674,0.2589,149.8970,220.4941,0.0491,0.3384,0.2256
1003,569.0000,0.1783,101.4402,0.2356,134.0382,162.6611,0.0462,0.3009,0.1892
1004,558.0000,0.1765,98.5104,0.2342,130.7111,154.8253,0.0897,0.2815,0.1540
ens,571.0000,0.2096,119.6918,0.2713,154.9341,214.8970,0.0392,0.3525,0.2332
TB seed,573.0000,0.1964,112.6669,0.2596,148.9045,195.8716,0.0518,0.3223,0.2006
std seed,9.6695,0.0210,13.2922,0.0271,17.2943,39.8714,0.0246,0.0293,0.0440
seed tệ nhất,558.0000,0.1765,98.5104,0.2342,130.7111,154.8253,0.0212,0.2815,0.1540


## 3. Test 2025–2026 (ngoài mẫu) với config đã khoá

In [7]:
P_test, R_test = st.walk_forward_blend(D, cfg_l, "test")
test = st.evaluate(D, P_test, cfg_l, "test")
st.save_result(test, cfg_l, "test_walkforward")
st.save_live_artifacts(test, cfg_l)        # backtest_trades.csv (trang public) + walkforward_signals.csv (kiểm thử live)
display(R_test)
display(pd.Series(test["summary"]))
display(test["by_year"])
display(test["by_quarter"])
st.seed_table(D, P_test, cfg_l, "test")

test 2025Q1: AUC touch 0.763 | AUC direction 0.552 | train 33,607 | 0s
test 2025Q2: AUC touch 0.729 | AUC direction 0.556 | train 35,815 | 0s
test 2025Q3: AUC touch 0.713 | AUC direction 0.563 | train 38,023 | 0s
test 2025Q4: AUC touch 0.758 | AUC direction 0.530 | train 40,183 | 0s
test 2026Q1: AUC touch 0.743 | AUC direction 0.570 | train 42,367 | 0s
test 2026Q2: AUC touch 0.741 | AUC direction 0.591 | train 44,575 | 0s
test 2026Q3: AUC touch 0.725 | AUC direction 0.564 | train 46,783 | 0s
test 2026Q4: AUC touch 0.726 | AUC direction 0.631 | train 48,943 | 0s
test 2025Q1: AUC touch 0.742 | AUC direction 0.560 | train 33,607 | 0s
test 2025Q2: AUC touch 0.706 | AUC direction 0.538 | train 35,815 | 0s
test 2025Q3: AUC touch 0.704 | AUC direction 0.489 | train 38,023 | 0s
test 2025Q4: AUC touch 0.727 | AUC direction 0.524 | train 40,183 | 0s
test 2026Q1: AUC touch 0.739 | AUC direction 0.557 | train 42,367 | 0s
test 2026Q2: AUC touch 0.732 | AUC direction 0.592 | train 44,575 | 0s
test 2

,quý,mẫu test,tỉ lệ chạm,b TB %,AUC touch,AUC touch của Direction,AUC direction (đã chạm),n train,cây touch TB,cây direction TB,giây,model
0,2025Q1,2160,0.469444,1.620745,0.762683,0.688546,0.551747,33607,342.4,176.6,0,xgb
1,2025Q2,2184,0.445055,1.233819,0.728808,0.625745,0.556446,35815,279.4,317.6,0,xgb
2,2025Q3,2208,0.403533,1.030739,0.713160,0.636096,0.562771,38023,323.2,747.8,0,xgb
3,2025Q4,2208,0.484149,1.369672,0.758110,0.653367,0.530385,40183,327.2,664.8,0,xgb
4,2026Q1,2160,0.517593,1.487610,0.742633,0.639842,0.569834,42367,188.8,385.4,0,xgb
5,2026Q2,2184,0.473901,1.232341,0.740787,0.646648,0.590563,44575,586.6,564.4,0,xgb
6,2026Q3,2208,0.422554,1.127097,0.725381,0.604406,0.564255,46783,359.6,781.8,0,xgb
7,2026Q4,179,0.519553,1.021163,0.725556,0.639035,0.630863,48943,464.6,460.0,0,xgb
8,2025Q1,2160,0.469444,1.620745,0.741997,0.645723,0.559749,33607,1.4,2.2,0,lstm
9,2025Q2,2184,0.445055,1.233819,0.706471,0.597117,0.538161,35815,1.4,2.4,0,lstm


trades                                                               376
win_rate                                                        0.614362
R_per_trade                                                     0.213966
total_R                                                        80.451175
max_dd_R                                                        9.502109
long_share                                                      0.492021
maker_share                                                     0.659574
timeout_share                                                   0.220745
b_mean_pct                                                      1.246537
pct_per_trade                                                   0.289414
total_pct                                                     108.819745
R_by_year               {2025: 35.488873289958, 2026: 44.96230173309978}
pct_by_year            {2025: 44.15826427723916, 2026: 64.66148054788...
trades_by_year                                    {

,lệnh,R/lệnh,tổng R,thắng,tỉ lệ LONG,TP,SL,hết giờ,khớp limit,b TB %,% giá/lệnh,tổng % giá,lãi % vốn
năm,,,,,,,,,,,,,
2025,199,0.178,35.489,0.613,0.412,0.538,0.281,0.181,0.658,1.203,0.222,44.158,39.400
2026,177,0.254,44.962,0.616,0.582,0.548,0.186,0.266,0.661,1.295,0.365,64.661,54.073


,lệnh,R/lệnh,tổng R,thắng,tỉ lệ LONG,TP,SL,hết giờ,khớp limit,b TB %,% giá/lệnh,tổng % giá,lãi % vốn
quý,,,,,,,,,,,,,
2025Q1,41,0.011,0.468,0.537,0.268,0.537,0.463,0.000,0.756,1.304,-0.085,-3.482,-0.237
2025Q2,60,0.411,24.669,0.750,0.800,0.650,0.167,0.183,0.583,1.142,0.468,28.078,27.076
2025Q3,47,-0.160,-7.513,0.404,0.213,0.298,0.362,0.340,0.723,1.051,-0.164,-7.704,-7.629
2025Q4,51,0.350,17.864,0.706,0.255,0.627,0.196,0.176,0.608,1.334,0.535,27.267,19.039
2026Q1,65,0.274,17.801,0.615,0.831,0.600,0.200,0.200,0.692,1.561,0.496,32.242,18.652
2026Q2,90,0.228,20.498,0.611,0.400,0.500,0.178,0.322,0.678,1.117,0.240,21.604,21.800
2026Q3,22,0.303,6.663,0.636,0.591,0.591,0.182,0.227,0.500,1.238,0.492,10.816,6.611


,lệnh,R/lệnh,tổng R,% giá/lệnh,tổng % giá,lãi % vốn,R/lệnh 2025,R/lệnh 2026
seed,,,,,,,,
1000,388.0000,0.2259,87.6405,0.3090,119.8987,130.3365,0.1455,0.3150
1001,392.0000,0.1974,77.3925,0.2609,102.2854,107.8487,0.1428,0.2567
1002,383.0000,0.1899,72.7262,0.2641,101.1383,98.4428,0.1540,0.2316
1003,384.0000,0.1826,70.1327,0.2384,91.5610,93.5840,0.1580,0.2089
1004,378.0000,0.1744,65.9121,0.2535,95.8375,85.2976,0.1005,0.2531
ens,376.0000,0.2140,80.4512,0.2894,108.8197,114.7776,0.1783,0.2540
TB seed,385.0000,0.1940,74.7608,0.2652,102.1442,103.1019,0.1402,0.2530
std seed,5.2915,0.0197,8.3167,0.0264,10.8170,17.2739,0.0230,0.0396
seed tệ nhất,378.0000,0.1744,65.9121,0.2384,91.5610,85.2976,0.1005,0.2089


In [8]:
# frozen: 1 bộ model train tới trước 2025 dùng cho cả giai đoạn test (đo độ bền khi không train lại)
P_fr, _ = st.walk_forward_blend(D, cfg_l, "frozen")
frozen = st.evaluate(D, P_fr, cfg_l, "frozen")
st.save_result(frozen, cfg_l, "test_frozen")
display(pd.Series(frozen["summary"]))
frozen["by_year"]

frozen 2025Q1: AUC touch 0.731 | AUC direction 0.557 | train 33,607 | 58s
frozen 2025Q1: AUC touch 0.707 | AUC direction 0.546 | train 33,607 | 150s


trades                                                               365
win_rate                                                         0.60274
R_per_trade                                                     0.198246
total_R                                                        72.359617
max_dd_R                                                       10.148526
long_share                                                      0.534247
maker_share                                                     0.706849
timeout_share                                                   0.254795
b_mean_pct                                                      1.253756
pct_per_trade                                                   0.265723
total_pct                                                      96.988713
R_by_year              {2025: 16.513871708013728, 2026: 55.8457450679...
pct_by_year            {2025: 18.94697799364358, 2026: 78.04173537080...
trades_by_year                                    {

,lệnh,R/lệnh,tổng R,thắng,tỉ lệ LONG,TP,SL,hết giờ,khớp limit,b TB %,% giá/lệnh,tổng % giá,lãi % vốn
năm,,,,,,,,,,,,,
2025,174,0.095,16.514,0.54,0.414,0.500,0.299,0.201,0.707,1.198,0.109,18.947,15.723
2026,191,0.292,55.846,0.66,0.644,0.539,0.157,0.304,0.707,1.305,0.409,78.042,71.659


## 4. Xuất bundle live (model của quý chứa nến cuối cùng, 5 seed; LSTM train trên GPU, live dự báo bằng onnxruntime)

In [9]:
path = st.export_bundle(D, cfg_l)
b = st.load_bundle(path)
print(path, b["trained_quarter"], b["direction_blend"], pd.to_datetime(b["data_end_ns"], utc=True), len(b["history"]), "giờ lịch sử")
# kiểm tra: dự báo của bundle (ONNX) trên 48 nến cuối = lịch sử đã lưu
import nn_models
k, L = 48, nn_models.history_rows(b["models"])
pt, _ = st.predict_bundle(b, D.ds.x[-k:], D.xd[-k:], D.ds.timestamp_ns[-k:])
pu = np.array([st.predict_bundle(b, D.ds.x[i:i + 1], D.xd[i - L + 1:i + 1], D.ds.timestamp_ns[i - L + 1:i + 1])[1][-1]
               for i in range(len(D.xd) - k, len(D.xd))])
h = b["history"].tail(k)
print("max |Δp_touch|", np.abs(pt - h.p_touch.to_numpy()).max(), " max |Δp_up|", np.abs(pu - h.p_up.to_numpy()).max())

c:\Users\thanh\BTC_PIPELINE\.venv\lib\site-packages\torch\onnx\symbolic_opset9.py:4279: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  warnings.warn(


bundle 2026Q4 seed 1000: cây touch 482, cây direction 394, lstm epoch 2 (onnx = torch, sai số 2.6e-05)


c:\Users\thanh\BTC_PIPELINE\.venv\lib\site-packages\torch\onnx\symbolic_opset9.py:4279: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  warnings.warn(


bundle 2026Q4 seed 1001: cây touch 369, cây direction 522, lstm epoch 2 (onnx = torch, sai số 2.8e-05)


c:\Users\thanh\BTC_PIPELINE\.venv\lib\site-packages\torch\onnx\symbolic_opset9.py:4279: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  warnings.warn(


bundle 2026Q4 seed 1002: cây touch 493, cây direction 436, lstm epoch 2 (onnx = torch, sai số 1.7e-05)


c:\Users\thanh\BTC_PIPELINE\.venv\lib\site-packages\torch\onnx\symbolic_opset9.py:4279: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  warnings.warn(


bundle 2026Q4 seed 1003: cây touch 333, cây direction 525, lstm epoch 1 (onnx = torch, sai số 8.5e-06)


c:\Users\thanh\BTC_PIPELINE\.venv\lib\site-packages\torch\onnx\symbolic_opset9.py:4279: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  warnings.warn(


bundle 2026Q4 seed 1004: cây touch 646, cây direction 423, lstm epoch 1 (onnx = torch, sai số 1.6e-05)
artifacts\1h_strategy_v2\models_final.joblib 2026Q4 {'xgb': 0.5, 'lstm': 0.5} 2026-10-08 10:00:00+00:00 2339 giờ lịch sử
max |Δp_touch| 6.386825934789897e-08  max |Δp_up| 0.0
